# Валидация симулированных reads — мышь (`ERP003950`)

Выравнивает paired-end reads на соответствующие template sequences с помощью bowtie2 и рассчитывает mapped, properly paired, mismatch и self-origin rates.


## Зависимости


In [ ]:
import os, sys, sysconfig, subprocess

_ENV_CANDIDATES = [
    "/data/user/epishkin/conda/envs/bcr_env",
    "/opt/conda/envs/bcr_env",
]
_CONDA_ENV = next((p for p in _ENV_CANDIDATES if os.path.isdir(p + "/bin")), _ENV_CANDIDATES[-1])
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [
    _CONDA_ENV + "/lib/python3.11/site-packages",
    _CONDA_ENV + "/lib/python3.12/site-packages",
    sysconfig.get_path("purelib"),
]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
os.makedirs(os.environ["XDG_CONFIG_HOME"], exist_ok=True)

print(f"Using env: {_CONDA_ENV}")
for tool in ("bowtie2", "bowtie2-build", "samtools"):
    path = subprocess.run(["which", tool], capture_output=True, text=True).stdout.strip()
    if not path:
        raise RuntimeError(f"'{tool}' not found in PATH.")
    print(tool, "->", path)


## Параметры


In [ ]:
from pathlib import Path

VOLUME = Path("/data/user/epishkin")
DATASET = "ERP003950"
SAMPLES = ["ERR346596", "ERR346597", "ERR346598", "ERR346599", "ERR346600", "ERR346601"]

SIM_VARIANT = "insilicoseq_150bp_pcr_fragmented"
SIM_DIR = VOLUME / "results" / DATASET / "simulated" / SIM_VARIANT
FASTQ_DIR = SIM_DIR / "fastq"

SHARED_TEMPLATES_DIR = VOLUME / "results" / DATASET / "simulated" / "insilicoseq" / "templates"

ALIGN_DIR = SIM_DIR / "alignment"
INDEX_DIR = ALIGN_DIR / "index"
BAM_DIR = ALIGN_DIR / "bam"
LOGS_DIR = ALIGN_DIR / "logs"
QC_DIR = ALIGN_DIR / "qc"
for d in (INDEX_DIR, BAM_DIR, LOGS_DIR, QC_DIR):
    d.mkdir(parents=True, exist_ok=True)

NPROC = 8
FORCE = False

print("SIM_DIR:", SIM_DIR)
print("ALIGN_DIR:", ALIGN_DIR)


## Индексы bowtie2

Для каждого sample индекс строится по его собственным templates.


In [ ]:
import time

def run_with_heartbeat(cmd, log_path, heartbeat=30, shell=False):
    t0 = time.time()
    with open(log_path, "w") as log_h:
        proc = subprocess.Popen(cmd, stdout=log_h, stderr=subprocess.STDOUT, text=True, shell=shell)
        label = cmd if shell else " ".join(cmd)
        print(f"[run] {label}\n  pid={proc.pid} log={log_path}")
        while proc.poll() is None:
            print(f"  still running: pid={proc.pid} elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(heartbeat)
    elapsed = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"command failed (exit {proc.returncode}); see {log_path}")
    print(f"done: elapsed={elapsed/60:.1f} min")


def build_index(sample, force=FORCE):
    templates_fa = SHARED_TEMPLATES_DIR / f"{sample}_templates.fasta"
    if not templates_fa.exists():
        raise FileNotFoundError(f"Missing templates for {sample}: {templates_fa}")
    index_prefix = INDEX_DIR / sample
    done_marker = Path(str(index_prefix) + ".1.bt2")
    if done_marker.exists() and not force:
        print(f"[{sample}] [skip] index exists: {index_prefix}")
        return index_prefix
    run_with_heartbeat(
        ["bowtie2-build", "--threads", str(NPROC), str(templates_fa), str(index_prefix)],
        LOGS_DIR / f"{sample}_bowtie2_build.log",
    )
    return index_prefix


In [ ]:
indexes = {sample: build_index(sample) for sample in SAMPLES}


## Выравнивание

Используется local alignment для учёта ошибок и indels на концах reads.


In [ ]:
def align_sample(sample, force=FORCE):
    bam_path = BAM_DIR / f"{sample}.bam"
    if bam_path.exists() and not force:
        print(f"[{sample}] [skip] BAM exists: {bam_path}")
        return bam_path

    r1 = FASTQ_DIR / f"{sample}_R1.fastq.gz"
    r2 = FASTQ_DIR / f"{sample}_R2.fastq.gz"
    if not (r1.exists() and r2.exists()):
        raise FileNotFoundError(f"Missing simulated fastq for {sample}: {r1} / {r2}")

    align_cmd = (
        f"bowtie2 --local -p {NPROC} -x {indexes[sample]} -1 {r1} -2 {r2} "
        f"2> {LOGS_DIR / f'{sample}_bowtie2_align.log'} "
        f"| samtools view -bS - "
        f"| samtools sort -@ {NPROC} -o {bam_path} -"
    )
    run_with_heartbeat(align_cmd, LOGS_DIR / f"{sample}_align_pipe.log", shell=True)
    subprocess.run(["samtools", "index", str(bam_path)], check=True)
    print(f"[{sample}] indexed {bam_path}")
    return bam_path


In [ ]:
bams = {sample: align_sample(sample) for sample in SAMPLES}


## Метрики

Собираются mapping statistics, mismatch rate и соответствие read исходному template.


In [ ]:
import re
import pysam

FLAGSTAT_MAPPED_RE = re.compile(r"(\d+) \+ \d+ mapped \(([\d.]+)")
FLAGSTAT_PAIRED_RE = re.compile(r"(\d+) \+ \d+ properly paired \(([\d.]+)")
STATS_ERROR_RATE_RE = re.compile(r"^error rate:\s+([\d.]+)")


def parse_flagstat(bam_path):
    out = subprocess.run(["samtools", "flagstat", str(bam_path)], capture_output=True, text=True, check=True).stdout
    mapped = FLAGSTAT_MAPPED_RE.search(out)
    paired = FLAGSTAT_PAIRED_RE.search(out)
    return {
        "mapped_pct": float(mapped.group(2)) if mapped else None,
        "properly_paired_pct": float(paired.group(2)) if paired else None,
    }


def parse_error_rate(bam_path):
    out = subprocess.run(["samtools", "stats", str(bam_path)], capture_output=True, text=True, check=True).stdout
    for line in out.splitlines():
        m = STATS_ERROR_RATE_RE.match(line)
        if m:
            return float(m.group(1))
    return None


def template_id_from_read_name(qname):
    # ISS read name: {template_id}_{i}_{j}/1 -- template_id itself ends in _n<count>,
    # so strip exactly the last two "_<int>" segments (before an optional /1 or /2, already gone).
    parts = qname.split("_")
    return "_".join(parts[:-2])


def self_origin_rate(bam_path, sample, max_reads=200_000):
    total = 0
    matched = 0
    with pysam.AlignmentFile(str(bam_path), "rb") as bam:
        for read in bam.fetch(until_eof=True):
            if read.is_unmapped or read.is_secondary or read.is_supplementary:
                continue
            total += 1
            if total > max_reads:
                break
            expected = template_id_from_read_name(read.query_name)
            if expected == read.reference_name:
                matched += 1
    return (matched / total) if total else None


In [ ]:
import csv

qc_rows = []
for sample in SAMPLES:
    bam_path = bams[sample]
    row = {"sample": sample, "bam": str(bam_path)}
    row.update(parse_flagstat(bam_path))
    row["error_rate"] = parse_error_rate(bam_path)
    row["self_origin_rate"] = self_origin_rate(bam_path, sample)
    qc_rows.append(row)
    print(sample, row)

qc_path = QC_DIR / "alignment_qc.tsv"
with open(qc_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(qc_rows[0].keys()), delimiter="\t")
    writer.writeheader()
    writer.writerows(qc_rows)
print(f"wrote {qc_path}")


## Интерпретация

Высокие mapped и self-origin rates подтверждают согласованность reads с исходными templates.
